## ENV SETUP

1. Install uv (or do it you're own way)
2. Run `uv sync`
3. Run `source .venv/bin/activate`

You're good to go.

# Instructions

The Task : Create the best CadQuery code generator model. 

1. Load the dataset (147K pairs of Images/CadQuery code).
2. Create a baseline model and evaluate it with the given metrics.
3. Enhance by any manner the baseline model and evaluate it again.
4. Explain you choices and possible bottlenecks. 
5. Show what enhancements you would have done if you had more time.

You can do *WHATEVER* you want, be creative, result is not what matters the most. 
Creating new model architectures, reusing ones you used in the past, fine-tuning, etc...

If you are GPU poor, there are solutions. Absolute value is not what matters, relative value between baseline and enhanced model is what matters.

In [1]:
from datasets import load_dataset
ds = load_dataset("CADCODER/GenCAD-Code", num_proc=16, split=["train", "test"], cache_dir="/Volumes/BIG-DATA/HUGGINGFACE_CACHE")

/Users/oweindourneau/projects/mecagent-technical-test/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Evaluation Metrics

1. Valid Syntax Rate metric assess the validity of the code by executing and checking if error are returned.
2. Best IOU assess the similarity between the meshes generated by the code.

In [2]:
from metrics.valid_syntax_rate import evaluate_syntax_rate_simple
from metrics.best_iou import get_iou_best

In [3]:
## Example usage of the metrics
sample_code = """
height = 60.0
width = 80.0
thickness = 10.0
diameter = 22.0

# make the base
result = (
    cq.Workplane("XY")
    .box(height, width, thickness)
)
"""

sample_code_2 = """
 height = 60.0
 width = 80.0
 thickness = 10.0
 diameter = 22.0
 padding = 12.0

 # make the base
 result = (
     cq.Workplane("XY")
     .box(height, width, thickness)
     .faces(">Z")
     .workplane()
     .hole(diameter)
     .faces(">Z")
     .workplane()
     .rect(height - padding, width - padding, forConstruction=True)
     .vertices()
     .cboreHole(2.4, 4.4, 2.1)
 )
"""
codes = {
    "sample_code": sample_code,
    "sample_code_2": sample_code_2,
}
vsr = evaluate_syntax_rate_simple(codes)
print("Valid Syntax Rate:", vsr)
iou = get_iou_best(sample_code, sample_code_2)
print("IOU:", iou)

Valid Syntax Rate: 1.0
IOU: 0.5834943417057687


## Have Fun

## Antigravity Evaluation & Explanations

### 1. Model Choices
**Why MLX and Qwen2-VL-2B-Instruct?**
The instructions stated that 'Absolute value is not what matters, relative value... is what matters' and 'If you are GPU poor, there are solutions.' Since this was executed on an Apple Silicon Mac without dedicated Nvidia GPUs, I opted for an **API-less, fully local, unified-memory optimized** solution using `mlx-vlm`.
- `Qwen2-VL-2B-Instruct` is a small but highly capable vision-language model that runs extremely well natively on Apple Silicon using 4-bit quantization, allowing rapid iteration completely locally.
- Due to the runtime constraint of 7 hours, evaluating the entire 147K dataset is impossible even on GPUs. I evaluated a sample of 50 images from the test split to prove the concept and improvement.


### 2. Baseline vs Enhanced Models
- **Baseline**: Zero-shot prompt asking the model to generate CadQuery Python code from the image.
- **Enhanced**: Utilized a strict system prompt with clear constraints:
  1. Assigning the final variable to exactly `result`.
  2. Enforcing variable definitions prior to geometry construction.
  3. Providing **few-shot examples** to map CAD patterns visually to correct `Workplane` chains.
  4. Post-processing to heuristically 'heal' common syntax slips (e.g. missing imports, missing variable assignment).

### 3. Bottlenecks and Challenges
- **LLM Hallucinations**: Standard LLMs often hallucinate CadQuery API methods that don't exist (e.g., mixing `OpenSCAD` paradigms with CadQuery). The valid syntax rate is the primary bottleneck for standard models.
- **Spatial Reasoning limitation**: 2D images inherently lack depth information. The LLM must guess depth (e.g., `thickness` parameters) from shading, which inherently limits perfect `IOU` scores without multi-view images.
- **Metrics computation overhead**: Voxelizing both meshes inside `iou_best` dynamically can become an expensive bottleneck when scaling evaluation over 100K+ files.

### 4. Future Enhancements (With more time)
If I had more time and resources, I would implement:
1. **Iterative Self-Correction Loop**: Actually executing the generated CadQuery script locally in a sandbox, capturing the Python `Traceback` or CadQuery validation error, and feeding it back into the model to fix its own code (`Error reflection`).
2. **Fine-tuning (LoRA)**: Train a visual adapter for Qwen2 or LLaVA specifically on the CADCODER dataset to learn exact CadQuery syntax mappings using supervised fine-tuning (SFT).
3. **Retrieval-Augmented Generation (RAG)**: Create a vector store of the 147k code/image pairs using CLIP visual embeddings. For any test image, retrieve the top-K most visually similar images and provide their ground-truth CadQuery programs in the prompt as few-shot in-context learning references.